In [82]:
import boto3
from botocore.exceptions import (
    NoCredentialsError,
    ClientError,
    BotoCoreError
)

In [83]:
class S3UtilsListFolders:

    @staticmethod
    def list_s3_folders(
        bucket_name,
        access_key,
        secret_key,
        region_name=None,
        max_folders=10
    ):
        """
        List up to max_folders top-level folders in an S3 bucket.

        Args:
            bucket_name (str):
                Name of the S3 bucket.

            access_key (str):
                AWS access key ID.

            secret_key (str):
                AWS secret access key.

            region_name (str, optional):
                AWS region containing the bucket. If None, boto3 uses
                its default region configuration.

            max_folders (int, optional):
                Maximum number of folders to return. Defaults to 10.

        Returns:
            tuple[bool, list[str] | str]:
                A tuple containing:

                - bool:
                    True if the folders were retrieved successfully,
                    otherwise False.

                - list[str] | str:
                    List of top-level folder prefixes on success,
                    or a descriptive error message on failure.
        """

        try:
            s3 = boto3.client(
                "s3",
                aws_access_key_id=access_key,
                aws_secret_access_key=secret_key,
                region_name=region_name
            )

            response = s3.list_objects_v2(
                Bucket=bucket_name,
                Delimiter="/",
                MaxKeys=max_folders
            )

            folders = [
                prefix["Prefix"]
                for prefix in response.get("CommonPrefixes", [])
            ]

            return True, folders

        except NoCredentialsError:
            return False, "AWS credentials are invalid or could not be found."

        except ClientError as e:
            error = e.response.get("Error", {})
            code = error.get("Code", "Unknown")
            message = error.get("Message", str(e))

            return False, f"AWS error {code}: {message}"

        except BotoCoreError as e:
            return False, f"AWS SDK error: {str(e)}"

        except Exception as e:
            return False, f"Unexpected error: {str(e)}"

In [84]:
class S3UtilsListFiles:

    @staticmethod
    def list_s3_files(
        bucket_name,
        access_key,
        secret_key,
        region_name=None,
        folder_prefix="",
        max_files=1
    ):
        """
        List up to max_files files from a specific S3 folder.

        Returns:
            tuple[bool, list[str] | str]
        """

        try:
            s3 = boto3.client(
                "s3",
                aws_access_key_id=access_key,
                aws_secret_access_key=secret_key,
                region_name=region_name
            )

            response = s3.list_objects_v2(
                Bucket=bucket_name,
                Prefix=folder_prefix
            )

            files = [
                obj["Key"]
                for obj in response.get("Contents", [])
                if not obj["Key"].endswith("/")
            ]

            # Limit the results to max_files
            files = files[:max_files]

            return True, files

        except NoCredentialsError:
            return False, "AWS credentials are invalid or could not be found."

        except ClientError as e:
            error = e.response.get("Error", {})
            code = error.get("Code", "Unknown")
            message = error.get("Message", str(e))

            return False, f"AWS error {code}: {message}"

        except BotoCoreError as e:
            return False, f"AWS SDK error: {str(e)}"

        except Exception as e:
            return False, f"Unexpected error: {str(e)}"

In [91]:
import boto3

from botocore.exceptions import (
    NoCredentialsError,
    ClientError,
    BotoCoreError
)


class S3UtilsReadJpegFiles:

    @staticmethod
    def read_s3_jpeg_files(
        bucket_name,
        access_key,
        secret_key,
        folder_name,
        file_names,
        region_name=None,
        max_files=1
    ):
        """
        Read JPEG file information from S3.

        Returns the filename and total size in bytes.

        max_files defaults to 1.
        """

        try:
            s3 = boto3.client(
                "s3",
                aws_access_key_id=access_key,
                aws_secret_access_key=secret_key,
                region_name=region_name
            )

            results = []

            for file_name in file_names[:max_files]:

                # Only process JPEG files
                if not file_name.lower().endswith((".jpg", ".jpeg")):
                    continue

                # Determine whether file_name is already
                # a complete S3 object key.
                if file_name.startswith(folder_name):
                    object_key = file_name
                else:
                    object_key = f"{folder_name}{file_name}"

                response = s3.head_object(
                    Bucket=bucket_name,
                    Key=object_key
                )

                total_bytes = response["ContentLength"]

                # Display only the filename, not the folder
                if file_name.startswith(folder_name):
                    display_name = file_name[len(folder_name):]
                else:
                    display_name = file_name

                results.append(
                    (display_name, total_bytes)
                )

            return True, results

        except NoCredentialsError:
            return False, "AWS credentials are invalid or could not be found."

        except ClientError as e:
            error = e.response.get("Error", {})
            code = error.get("Code", "Unknown")
            message = error.get("Message", str(e))

            return False, f"AWS error {code}: {message}"

        except BotoCoreError as e:
            return False, f"AWS SDK error: {str(e)}"

        except Exception as e:
            return False, f"Unexpected error: {str(e)}"

In [92]:
bucket_name = "aaa"
access_key = "xxx"
secret_key = "yyy"

folders = S3UtilsListFolders.list_s3_folders(
    bucket_name,
    access_key,
    secret_key,
    region_name=None,
    max_folders=10
)

for folder in folders:
    print(folder)

False
AWS error InvalidAccessKeyId: The AWS Access Key Id you provided does not exist in our records.


In [93]:
bucket_name = "xx"
access_key = "xx"
secret_key = "xx"

folders = S3UtilsListFolders.list_s3_folders(
    bucket_name,
    access_key,
    secret_key,
    region_name=None,
    max_folders=10
)
print()
for folder in folders:
    print(folder)


True
['8472714338/', '8479755334/']


In [88]:
bucket_name = "xx"
access_key = "xx"
secret_key = "xx"

success, folders = S3UtilsListFolders.list_s3_folders(
    bucket_name=bucket_name,
    access_key=access_key,
    secret_key=secret_key,
    region_name=None,
    max_folders=10
)
print(success)
print(folders)
if success:

    for folder in folders:

        success, files = S3UtilsListFiles.list_s3_files(
            bucket_name=bucket_name,
            access_key=access_key,
            secret_key=secret_key,
            region_name=None,
            folder_prefix=folder,
            max_files=3
        )

        if success:

            for file in files:
                # Remove the folder prefix from the S3 key
                file_name = file[len(folder):]

                print(f'"{folder}","{file_name}"')

        else:
            print(f'Error listing files in "{folder}": {files}')

else:
    print(f'Error listing folders: {folders}')

True
['8472714338/', '8479755334/']
"8472714338/","8472714338/201305__/IMG_0073.JPG"
"8472714338/","8472714338/201305__/IMG_0075.JPG"
"8472714338/","8472714338/201305__/IMG_0076.JPG"
"8479755334/","201607__/IMG_1872.JPG"
"8479755334/","201908__/DADY6820.JPG"
"8479755334/","201908__/QNNL5903.JPG"


In [95]:
success, folders = S3UtilsListFolders.list_s3_folders(
    bucket_name=bucket_name,
    access_key=access_key,
    secret_key=secret_key,
    region_name=None,
    max_folders=10
)

if success:

    for folder in folders:

        success, files = S3UtilsListFiles.list_s3_files(
            bucket_name=bucket_name,
            access_key=access_key,
            secret_key=secret_key,
            region_name=None,
            folder_prefix=folder,
            max_files=3
        )

        if success:

            success, results = S3UtilsReadJpegFiles.read_s3_jpeg_files(
                bucket_name=bucket_name,
                access_key=access_key,
                secret_key=secret_key,
                folder_name=folder,
                file_names=files,
                region_name=None,
                max_files=3
            )

            if success:

                for file_name, total_bytes in results:
                    print(f'"{file_name}", {total_bytes} bytes')

            else:
                print(f"Error reading files: {results}")

        else:
            print(f'Error listing files in "{folder}": {files}')

else:
    print(f"Error listing folders: {folders}")

"8472714338/201305__/IMG_0073.JPG", 1116347 bytes
"8472714338/201305__/IMG_0075.JPG", 1096620 bytes
"8472714338/201305__/IMG_0076.JPG", 1194088 bytes
"201607__/IMG_1872.JPG", 255708 bytes
"201908__/DADY6820.JPG", 33169 bytes
"201908__/QNNL5903.JPG", 36546 bytes
